# DOI errors
This notebook fixes DOI errors
Checked for the following error types in a given list 
1. Included resolver (https://doi.org/, or http://doi.org/) at beginning or end of DOI 
2. Two DOIs concatenated together 
3. Forward slash encoded as ‘%2F’ 
4. Ending with period (.) 
5. Ending or starts with forward slash (/) 
6. Ending with space (and other characters after space) 
7. Ending with characters not part of DOI 
8. Contains NA or NF 
9. If unregistered in a RA 
10. Incomplete 
11. Badly formed prefix or suffix (missing or changed characters) 
12. Contains Crossref test account prefix
13. Duplicated DOIS within our dataset
14. Begins with 'doi:' or 'DOI:'

## UPDATED APRIL 10 2025
- [x] reran data.
- [x] cleaned up code. 

In [25]:
import pandas as pd
from pandas import DataFrame, Series
import re # for regex
import os
import json # for import
import requests # send API calls
import time # enable pause for API calls
import numpy as np
from colorama import Fore,Style,Back,init # just for fun
from typing import List,Any

# set display options
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None) # show all columns
pd.set_option('display.max_colwidth', None)  # Show full content of each column




In [4]:
# import raw data

file = "IPCC_Report.txt"
df = pd.read_csv(file, sep='\t', encoding='utf-8', header=0,na_values=['NA','NA'])
print(df.columns)
#df

Index(['doi'], dtype='object')


In [5]:
# characterize df
# total rows
print(f"Total rows: {len(df)}")

# Count NaN values
nan_count = df['doi'].isna().sum()
print(f"Number of NaN values in 'doi' column: {nan_count}")

# percentage of filled values
df_non_nan = (df['doi'].notna().sum()/len(df))*100
print(Fore.YELLOW + f"Percent that are not NaN: {df_non_nan:.1f}")

# lengths of DOIs
doi_lengths = df['doi'].str.len().describe()
print(Fore.CYAN + f"\nDOI Lengths: \n{doi_lengths}")

#reset colors
print(Style.RESET_ALL)


Total rows: 6135
Number of NaN values in 'doi' column: 0
Percent that are not NaN: 100.0

DOI Lengths: 
count    6135.000000
mean       23.652649
std         4.785057
min        13.000000
25%        20.000000
50%        24.000000
75%        27.000000
max        61.000000
Name: doi, dtype: float64



# #1 & #2: include resolver at beginning, end, or in middle of DOI
### how many rows start with 'https:' or 'http:'?

In [6]:
# count how many start with the resolver
count = df['doi'].str.startswith(('https://doi.org/','http://doi.org/')).sum()
print(f"Number of rows with http or https resolver: {count}")
percent = count/len(df)*100
print(f"percent with resolvers: {percent:.2f}%")

# Get rows with 'https:' with option for NaN value - passes a tuple for https or http
https_rows = df[df['doi'].str.startswith(('https:','http:'), na=False)]
print("\nRows starting with 'https:' or 'http':", len(https_rows))
percent = len(https_rows)/len(df)*100
print(f"Percentage of total: {percent:.2f}%")

print("\nThe first 5 examples:")
# Get just the 'doi' values that start with 'https:'
mask = (df['doi'].notna()) & (df['doi'].str.startswith(('https:','http:')))
https_rows = df[mask]['doi'].head(5)
print(https_rows)



Number of rows with http or https resolver: 1
percent with resolvers: 0.02%

Rows starting with 'https:' or 'http': 1
Percentage of total: 0.02%

The first 5 examples:
167    https://doi.org/10.2166/wp.2016.185
Name: doi, dtype: object


### now, let's remove https or http variants
This removes the resolver from anywhere in the string. When I checked a few manually, the removal of the resolver from the middle seems to create a valid DOI.

In [7]:
search_strings = ["https://doi.org/",
            "http://doi.org",
            "http://dx.doi.org/",
            
            ]
# remove https and http from the beginning - note the regex ^
# pattern = "^"+"|".join(search_strings)

# the following pattern just removes the search strings from anywhere in the 'doi'
pattern = "|".join(search_strings)
df['doi'] = df['doi'].str.replace(pattern, '', regex=True)



In [8]:
df['doi'].head(10)

0                     10.1002/rog.20022
1                   10.1038/nature19082
2                      10.1038/ngeo1787
3          10.1016/j.envdev.2018.12.003
4    10.1016/B978-0-12-407668-6.00010-0
5                  10.1038/Nclimate1666
6             10.1007/s10584-008-9520-z
7                       10.1002/wcc.297
8    10.1111/j.1467-7660.1995.tb00560.x
9             10.1080/10398560701701288
Name: doi, dtype: object

### Rows starting with 'https:' or 'http': 4675
### Percentage of total: 33.93%
All were corrected with removal. 

# 3. Forward slash encoded as ‘%2F’ 
- [x] identify those with encoding for forward slash
- [x] manually check if replacing encoding with literal forward slash will create a resolving DOI
- [x] replace encoding with literal forward slash

In [9]:
print("Those with '%2F' as encoding:")
# Get just the 'doi' values that contain the encoding
mask = (df['doi'].notna()) & (df['doi'].str.contains(('%2F')))
encoded_rows = df[mask]['doi']
print(encoded_rows)
print(f"\nThose with encoded forward slashes: {len(encoded_rows)}")

Those with '%2F' as encoding:
Series([], Name: doi, dtype: object)

Those with encoded forward slashes: 0


In [39]:
# replace encoded '%2F' with literal forward slash
df['doi'] = df['doi'].str.replace("%2F", '/', regex=False)

# test those indices that came up before: 6624, 6815, 11130
#df.iloc[11130]

# #4: Contains period at end of DOI
- [x] check for period at end of DOI. This is often a mistake in copy and paste, however, it is a legitimate character permitted in a DOI. 
- [x] manually check for legit DOI before correcting
- [x] correct any that are not legit.

In [10]:
# check for period at end of DOI
print("Those with literal period at end of DOI:")
# Get just the 'doi' values that end with .
mask = (df['doi'].notna()) & (df['doi'].str.endswith(('.')))
period_ending_rows = df[mask]['doi']
print(period_ending_rows)
print(f"\nThose with literal periods at end of the DOI: {len(period_ending_rows)}")

Those with literal period at end of DOI:
Series([], Name: doi, dtype: object)

Those with literal periods at end of the DOI: 0


In [11]:
# check to see if these are legit
def check_doi(doi):
    url = "https://api.crossref.org/works/"
    try:
        response = requests.get(f"{url}/{doi}&mailto:pnriddle@dal.ca")
        if response.status_code == 404:
            return "Resource not found"
        elif response.status_code == 200:
            return "found"
        else:
            return f"status code: {response.status_code}"
    except Exception as e:
        return f"Error: {str(e)}"

#apply to df2
df2 = pd.DataFrame(columns = ['status'])
df2['status'] = period_ending_rows.apply(check_doi)

In [12]:
print(f"Those with periods at end AND were not found in Crossref: {(len(df2))}")
df2

Those with periods at end AND were not found in Crossref: 0


,status


In [13]:
df['doi'] = df['doi'].str.rstrip(".")
# test on one of the indexes above
df.iloc[4224]

doi    10.1175/JPO-D-15-0056.1
Name: 4224, dtype: object

# # 5 & 6: Ending with forward slash (/) and Ending with space (and other characters after space) & 7 ending with chacters not part of the DOI. (partially)
- [x] identify those with endings
- [x] manually verify their legitimacy
- [x] run Crossref check
- [x] remove unwanted characters. 

### note
#7 is only partially complete. It will require manual checking of those that fail the registration check to see if there are obvious ones where characters, (such as error messages) have been concatenated onto a valid DOI.


In [14]:
# check for period at end of DOI
# Get just the 'doi' values that end with /
mask = (df['doi'].notna()) & (df['doi'].str.endswith(('/')) | df['doi'].str.startswith(('/')) )
slash_ending_rows = df[mask]['doi']
print(slash_ending_rows)
print(f"\nThose with literal slashes at beginning or end of the DOI: {len(slash_ending_rows)}")

print("\nThose ending with a space or containing a space in the DOI")
#get values that have or end with a space
mask = (df['doi'].notna()) & ((df['doi'].str.contains(' '))|(df['doi'].str.endswith(' ')))
contains_space = df[mask]['doi']
print(contains_space)
print(f"Those with containing or end with a space: {len(contains_space)}")

#how does it compare with those that end with a space


Series([], Name: doi, dtype: object)

Those with literal slashes at beginning or end of the DOI: 0

Those ending with a space or containing a space in the DOI
Series([], Name: doi, dtype: object)
Those with containing or end with a space: 0


In [15]:
# remove all / from 'doi' column
df['doi'] = df['doi'].apply(lambda x: x.lstrip('/') if isinstance(x,str) else x)


In [16]:
# these 9 need to be handled manually. I have created a list of tuples
# containing the index and the valid DOI value. 
tuple_list = [(657, "10.1136/bmjopen-2011-000152"),
                (1444, "10.33137/cjal-rcbu.v5.32938"),
                (1648, "10.7202/1021928ar"),
                (1649, "10.7202/1023786ar"),
                (1650, "10.7202/1030780ar"),
                (1651, "10.7202/1060828ar"),
                (4155, "10.7202/1033070"),
                (4331, "10.7202/1054201ar"),
                (8500, "10.1197/jamia.M1752")
                ]

for index, doi_value in tuple_list:
    df.at[index, 'doi'] = doi_value

In [48]:
# let's check those again:
df.iloc[657]['doi'] # yeah! that worked. 

'10.1136/bmjopen-2011-000152'

# #8 Contains NA or NF
- [ ] check those that are Nan
- [ ] check strings that contain NA or NF
- [ ] be consistent, replace NA and NF with NaN

In [17]:
# Count NaN values
nan_count = df['doi'].isna().sum()
print(f"Number of NaN values in 'doi' column: {nan_count}")
print(Fore.CYAN + f"Percentage that are NaN values: {(nan_count/len(df))*100:.1f}")

# percentage of filled values
df_non_nan = (df['doi'].notna().sum()/len(df))*100
print(Fore.YELLOW + f"Percent that are not NaN: {df_non_nan:.1f}")

# count of those that contain 'NA' or 'NF'
mask = ((df['doi'].str.startswith('NA'))|(df['doi'].str.startswith('NF')))
contains_nanf = df[mask]['doi'].describe()
print(contains_nanf)

print(Style.RESET_ALL)

Number of NaN values in 'doi' column: 0
Percentage that are NaN values: 0.0
Percent that are not NaN: 100.0
count       0
unique      0
top       NaN
freq      NaN
Name: doi, dtype: object



In [98]:
# replace 'NF' with np.nan
df['doi'] = df['doi'].replace('NF', np.nan)

## #11: badly formed prefix or suffix

- [x] replace those with periods instead of forward slashes<br>
- [x] replace those with forward slash encoded as %2F


In [18]:
# how many DOIs are invalid in their structure?

# set up function to catch those with sequential dots
def filter_invalid_dois(df: pd.DataFrame, doi_column: str = 'doi') -> pd.Series:
    # Pattern matches DOIs with multiple sequential dots like 10.1.1.86.4340
    invalid_pattern: str = r'^10\.\d+\.\d+\.\d+'
    
    # Return DOIs that don't match this pattern
    return df[df[doi_column].str.contains(invalid_pattern, na=False)]['doi']

invalid_dois = filter_invalid_dois(df,'doi')
print(len(invalid_dois)) 
# just 5 of them, and only 2 seem legitimately bad
invalid_dois



0


Series([], Name: doi, dtype: object)

In [52]:
# overwrite index 3 and 5 with ''
df.loc[3, 'doi'] = np.nan
df.loc[5, 'doi'] = np.nan
# then rerun above and check the remainder for legitimacy. 

# #12 Contain test prefixes
This checks for test prefixes 

In [19]:
# check which DOIs are on a test prefix, such as 10.5555, 10.505050
# from API calls:
# https://api.crossref.org/members?query=test%20accounts
# https://api.crossref.org/members?query=Crossref

# removed 10.13003 and 10.18810

test_prefixes = ["10.5555",
                "10.88888",
                "10.30444",
                "10.30446",
                "10.30447",
                "10.30448",
                "10.30449",
                "10.50505",
                "10.30443"]

# check if prefixes are in 'doi'
# just playing with type annotation here
def get_strings_in_column(
    df: DataFrame,
    column: str,
    search_strings: List[str],
    case: bool = False,
    regex: bool = False) -> Series:
    pattern: str = '|'.join(search_strings)
    return df[df[column].str.contains(
        pattern,
        case=case,
        regex=regex,
        na=False
    )]

In [20]:

# If you want the entire DataFrame rows that match:
mask = df['doi'].str.contains('|'.join(test_prefixes),na=False)
matching_rows: DataFrame = df[mask]

print(Fore.MAGENTA + f"number of matching rows: {len(matching_rows)}")
print(Style.RESET_ALL)
matching_rows


# If you want just the matching values from the specified column:
#matching_values: Series = df[df['doi'].str.contains('|'.join(test_prefixes), na=False)]['doi']

number of matching rows: 0



,doi


In [77]:
# so those will come back as invalid. I propose we overwrite them 
# and then search on their titles in Crossref or OpenAlex or manually search as there is only 7!
indexes_to_change = matching_rows.index

print(indexes_to_change)

Index([1553, 4996, 5049, 5058, 5219, 6225, 6487], dtype='int64')


In [132]:
# overwrite these indexes with None or valid doi
# Then use indexes to overwrite values
df.loc[mask,'doi'] = np.nan

# test to see if it worked
df.iloc[1553]


pub_id                                                      1771.0
title               Assessing the DVD Option in a Medical Library.
doc_type                                                   article
author_list_full                                        BD Cameron
publication_year                                            1998.0
source_name                                 Computers in libraries
volume                                                          18
issue                                                           10
pages                                                        60-64
bk_editor                                                      NaN
bk_edition                                                     NaN
publisher                                                      NaN
doi                                                            NaN
openalex_work_id                                               NaN
isbn                                                          

In [21]:
# Verify that no matching prefixes remain
mask_check = df['doi'].str.contains('|'.join(test_prefixes), na=False)
print(f"Check to see if any remain: {df[mask_check]}") 

Check to see if any remain: Empty DataFrame
Columns: [doi]
Index: []


# #13 duplicate DOIs
- [x] identify duplicate DOIs rows
- [x] identify if title is the same or different


In [26]:
DataFrame = pd.DataFrame # set type annotation

# For DOIs and titles that are duplicated
doi_dupes = df[df['doi'].notna() & df['doi'].duplicated(keep=False)].sort_values('doi')
print(Fore.CYAN + f"Number of duplicates: {len(doi_dupes)}")
print(Style.RESET_ALL)
print("Duplicated DOIs with their indices:")
print(doi_dupes[['doi']].to_string())

# drop duplicates but keep first occurrence
df:DataFrame = df.drop_duplicates(subset=['doi'], keep='first')
doi_dupes_deduped:DataFrame = doi_dupes.drop_duplicates(subset=['doi'], keep='first')

print(f"new length of df: {len(df)}")
print(f"number of DOIs that had duplicates: {len(doi_dupes_deduped)}")
print(f"this process removed: {len(doi_dupes)-len(doi_dupes_deduped)}")

Number of duplicates: 484

Duplicated DOIs with their indices:
                                        doi
1479                   10.1002/2013GL059010
2980                   10.1002/2013GL059010
2083                   10.1002/2013GL059069
3404                   10.1002/2013GL059069
3683                   10.1002/2014EF000272
6006                   10.1002/2014EF000272
2272                   10.1002/2014GL060140
3549                   10.1002/2014GL060140
4481                   10.1002/2014JC010470
1701                   10.1002/2014JC010470
666                    10.1002/2015GB005237
1664                   10.1002/2015GB005237
1254                   10.1002/2015GL063846
3983                   10.1002/2015GL063846
4323                   10.1002/2015gb005338
5571                   10.1002/2015gb005338
6057                   10.1002/2016EF000505
3761                   10.1002/2016EF000505
3396                   10.1002/2016GL067695
2078                   10.1002/2016GL067695
5111         

# #14 those starting with DOI: or doi:
- [x] identify how many start with 'DOI:' or 'doi:'
- [x] remove them using lstrip or replace


In [31]:
# identify how many start with 'DOI:' or 'doi:'

search_strings = ["DOI:","doi:"]
# remove https and http from the beginning - note the regex ^
pattern = "^"+"|".join(search_strings)
# the following pattern just removes the search strings from anywhere in the 'doi'
#pattern = "|".join(search_strings)

mask = df['doi'].str.contains(pattern,regex=True, na=False)
print(Fore.MAGENTA + f"How many DOIs contain 'DOI:' or 'doi:': {len(df[mask])}") 
print(Style.RESET_ALL)

How many DOIs contain 'DOI:' or 'doi:': 0



In [205]:
# remove those that match the pattern in the above cell:
df['doi'] = df['doi'].str.replace(pattern, '', regex=True)

# #9: check if DOI is valid/registered
- [ ] run through agency check
- [ ] run through each agencies validity check

In [27]:
from tqdm import tqdm
tqdm.pandas()

# check agency function
def agency_check(doi):
    """
    This function takes a DOI and passes it to a Crossref REST API call
    that checks which agency the DOI, if valid, is registered to. 
    Returns: Resource not found
            the agency label
            error code
    """
    url = "https://api.crossref.org/works/"
    try:
        response = requests.get(f"{url}{doi}/agency")
        if response.status_code == 404:
            return "Resource not found"
            print("Resource not found")
        elif response.status_code == 200:
            data = response.json()
            agency_id = data['message']['agency']['label']
            return agency_id
            print(agency_id)
        else:
            return f"Status code: {response.status_code}"
            print(f"error: {response.status_code}")
    except Exception as e:
        return f"There's some sort of error: {str(e)}"



def crossref_check_doi(doi):
    """
    This function takes a DOI and passes it to a Crossref REST API call
    returns: Resource not found
            ok status code to indicate it is registered and a valid DOI
            an error message
    """
    url = "https://api.crossref.org/works/"
    try:
        response = requests.get(f"{url}{doi}")
        if response.status_code == 404:
            return "Resource not found"
        elif response.status_code == 200:
            data = response.json()
            status_msg = data['status']
            return status_msg
        else:
            return f"status code: {response.status_code}"
    except Exception as e:
        return f"Error: {str(e)}"



In [28]:
doi = "10.4242/BalisageVol8.Huitfeldt02"
agency_check(doi)

'Crossref'

In [29]:
#df2 = pd.DataFrame(columns=['registration_agency'])
# check which agency they are on#=
# df2['registration agency'] = df['doi'].apply(agency_check)

# If applying to a DataFrame column
# df2['agency'] = [agency_check(doi) for doi in tqdm(df['doi'], desc="Checking DOIs")]

# OR if applying using apply()
df['registration_agency'] = [agency_check(doi) if pd.notna(doi) else None 
                for doi in tqdm(df['doi'], desc="Checking agency for DOIs")]

Checking agency for DOIs: 100%|██████████| 5883/5883 [23:30<00:00,  4.17it/s]  


In [30]:
print(df.columns)
df['registration_agency'].value_counts()

Index(['doi', 'registration_agency'], dtype='object')


registration_agency
Crossref              5845
Resource not found      29
DataCite                 7
ISTIC                    2
Name: count, dtype: int64

All DOIs and their DOI registration status:<br>
|registration_agency| Count |
| ----------------- | ------|
|Crossref           |   9813|
|DataCite           |    280|
|Resource not found |    110|
|mEDRA              |      7|
|CNKI               |      1|
|ISTIC              |      1|
|KISTI              |      1|
<br>
Any with resource not found should be flagged for follow up with Crossref, title match with OpenAlex or OpenAire<br>


# # 10 Incomplete
Will need to manually check those that returned from #9 to see if incompleteness can be solved? May need to be a title and author match

In [32]:
df_incomplete = df[['doi','registration_agency']]
df_incomplete = df_incomplete.query("registration_agency == 'Resource not found'")
df_incomplete


,doi,registration_agency
108,10.1038/s41558-0190435-7,Resource not found
113,10.5194/tc-7375-2013,Resource not found
162,10.1890/1051-0761(2000)010[1270:UTEKIS]2.0.CO;2,Resource not found
166,10.1175/1520-0442(1999)012<2169:TDROTG>2.0.CO;2,Resource not found
178,10.5194/bg-113647-2014,Resource not found
261,10.1175/1520-0477(2000)081<0443:IOEWAC>2.3.CO;2,Resource not found
424,10.1659/0276-4741(2006)26[304:SMTAT]2.0.CO;2,Resource not found
883,10.1659/0276-4741(2004)024[0024:NHATNF]2.0.CO;2,Resource not found
1017,10.1002/(SICI)1099-1530(199610)7:4<301::AID-PPP231>3.0.CO;2-R,Resource not found
1228,10.1088/1748-9326/aac2f0/meta,Resource not found


In [33]:
### IT IS NOT NECESSARY TO RUN THIS CODE ###

results = []
for doi in tqdm(df['doi'], desc="Checking Crossref for DOIs"):
    if pd.notna(doi):
        check_result = crossref_check_doi(doi)
        results.append({'doi': doi, 'crossref_registration_check': check_result})
    else:
        results.append({'doi': None, 'crossref_registration_check': None})
df2 = pd.DataFrame(results)

#then join df2 to df on doi
print(f"length of df2: {len(df2)}")
print(f"length of df: {len(df)}")

Checking Crossref for DOIs: 100%|██████████| 5883/5883 [22:05<00:00,  4.44it/s]  


In [38]:
# check those that are resource not found and don't belong to Crossref
print(Fore.CYAN + "DOIS and their registration agencies after cleaning:")
print("These numbers may change as publishers register or transfer DOIs.")
print(Fore.YELLOW + f"{df['registration_agency'].value_counts()}")

print(df3.dtypes)

print(Style.RESET_ALL)



DOIS and their registration agencies after cleaning:
These numbers may change as publishers register or transfer DOIs.
registration_agency
Crossref              5845
Resource not found      29
DataCite                 7
ISTIC                    2
Name: count, dtype: int64
doi                            object
registration_agency            object
crossref_registration_check    object
dtype: object



In [37]:
#export as .txt
file_to_be_saved = "IPCC_Report_cleaned_DOIs.txt"
df.to_csv(file_to_be_saved, sep='\t', encoding='utf-8',na_rep='NA')

All DOIs and their DOI registration status:<br>
|registration_agency| Count |
| ----------------- | ------|
|Crossref           |   9813|
|DataCite           |    280|
|Resource not found |    110|
|mEDRA              |      7|
|CNKI               |      1|
|ISTIC              |      1|
|KISTI              |      1|
<br>
Any with resource not found should be flagged for follow up with Crossref, title match with OpenAlex or OpenAire<br>
